In [6]:
import os
import json
from typing import Dict, Any, List
from dotenv import load_dotenv

# --- LangChain Imports ---
from langchain_community.chat_models import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.runnables import RunnablePassthrough # Used for creating the chain

# --- Project Imports ---
from utils.models import GraphState, KnowledgeGraphSchema

In [7]:
load_dotenv()

# --- 1. Define the Model and Parser ---

# The parser uses the final schema we defined in models.py
parser = JsonOutputParser(pydantic_object=KnowledgeGraphSchema)

# Initialize the LLM (Ensure Ollama is running with 'llama3')
# Set temperature to 0.0 for deterministic, fact-based extraction
llm = ChatOllama(model="llama3", temperature=0.0)

# --- 2. Define the Prompt Template ---

# This highly-detailed prompt is essential for guiding the LLM to output clean, structured JSON.
SYSTEM_PROMPT = """
You are an expert Knowledge Graph extractor. Your task is to analyze the provided scientific text 
and extract entities (Nodes) and their relationships (Relationships) strictly based 
on the JSON schema provided below.

# Instructions:
1. **Nodes (Entities):** Extract concepts like Genes, Proteins, Diseases, and Drugs relevant to Parkinson's research.
   - The 'id' must be a concise, unique name (e.g., 'LRRK2Gene', 'ParkinsonsDisease'). Use CamelCase.
   - The 'type' must be the high-level category (e.g., 'Gene', 'Disease', 'Compound').
2. **Relationships:** Identify directed relationships between the extracted Nodes.
   - 'source_id' and 'target_id' MUST match the 'id' of an extracted Node exactly.
   - 'type' MUST be uppercase with underscores (e.g., 'TARGETS', 'CAUSES_RISK', 'MITIGATES_EFFECT').
3. **Strict Compliance:** Your output MUST be a single, valid JSON object that strictly adheres 
   to the provided JSON schema. Do not include any explanatory text or markdown outside the JSON block.

# JSON Schema:
{format_instructions}
"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", SYSTEM_PROMPT),
        ("user", "Extract the Knowledge Graph from this text: \n\n--- TEXT ---\n\n{text}"),
    ]
)

# --- 3. Define the LLM Chain ---
# The chain executes in this order: Prompt -> LLM (Llama 3) -> Parser (Pydantic/JSON)
kg_extraction_chain = (
    prompt 
    | llm
    | parser
)

# --- 4. Define the Agent Function (LangGraph Node) ---

def extract_schema_from_text(state: GraphState) -> Dict[str, Any]:
    """
    LangGraph node function to extract Knowledge Graph data from text using the LLM chain.
    """
    print("\n--- NODE: SCHEMA ALIGNER STARTING (Running Llama 3) ---")
    
    text = state["text"]
    
    try:
        # Invoke the chain, passing the extracted text and the schema instructions
        result = kg_extraction_chain.invoke(
            {
                "text": text,
                "format_instructions": parser.get_format_instructions(),
            }
        )
        
        # Convert the Pydantic models back into standard dictionaries for storage in GraphState
        extracted_data_dict = {
            "nodes": [node.model_dump() for node in result.nodes],
            "relationships": [rel.model_dump() for rel in result.relationships],
        }
        
        # Update and return the state
        return {
            "graph_data_list": [extracted_data_dict],
            "is_valid": True,
            "error_message": None,
            "attempts": state["attempts"] + 1,
        }

    except Exception as e:
        print(f"--- ERROR IN EXTRACTION: {e} ---")
        # If the LLM output is bad JSON, we mark the state as invalid
        return {
            "is_valid": False,
            "error_message": str(e),
            "attempts": state["attempts"] + 1,
        }